# Multilinear Algebra & Tensors
## From Multi-Way Arrays to Decompositions and Robotics Applications

This notebook provides a self-contained introduction to **tensors** (multi-way arrays) and **multilinear algebra** --- the mathematical framework that extends matrices to higher dimensions. We develop the theory from first principles and implement key algorithms from scratch.

**What you'll learn:**
1. Tensor fundamentals: order, modes, fibers, slices, unfoldings
2. Tensor products and multilinear maps
3. CP (CANDECOMP/PARAFAC) decomposition via alternating least squares
4. Tucker decomposition and Higher-Order SVD (HOSVD)
5. Tensor contraction and Einstein summation notation
6. Application: inertia tensors for rigid body dynamics
7. Application: manipulability tensors for robot arms

**Prerequisites:** Linear algebra (matrix factorizations, SVD, eigendecomposition), basic multivariable calculus.

**References:**
- Kolda & Bader, *Tensor Decompositions and Applications*, SIAM Review, 2009.
- De Lathauwer, De Moor & Vandewalle, *A Multilinear Singular Value Decomposition*, SIAM J. Matrix Anal. Appl., 2000.
- Cichocki et al., *Tensor Decompositions for Signal Processing Applications*, IEEE Signal Processing Magazine, 2015.

---
## 1. Introduction: Beyond Matrices

In many applications, data naturally has more than two dimensions:

| Domain | Data Structure | Dimensions |
|--------|---------------|------------|
| **Video** | Pixel intensities | height $\times$ width $\times$ frames |
| **Chemometrics** | Fluorescence spectra | excitation $\times$ emission $\times$ samples |
| **Neuroscience** | Brain activity | neurons $\times$ time $\times$ trials |
| **Robotics** | Joint torque coupling | joints $\times$ joints $\times$ configuration |
| **Mechanics** | Stress-strain | spatial dim $\times$ spatial dim $\times$ material axes |

A **tensor** (in the sense of multilinear algebra) is a multi-dimensional array that generalizes vectors (1D) and matrices (2D) to arbitrarily many dimensions. The number of dimensions is called the **order** (or **mode count**):

$$\text{Scalar} \xrightarrow{\text{order 0}} \text{Vector} \xrightarrow{\text{order 1}} \text{Matrix} \xrightarrow{\text{order 2}} \text{3rd-order tensor} \xrightarrow{\text{order 3}} \cdots$$

While NumPy's `ndarray` handles multi-dimensional storage, the key insight of multilinear algebra is that tensors have **structure** that can be exploited through decompositions --- just as SVD reveals the structure of matrices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# Tensor dimensions for examples
TENSOR_SHAPE = (4, 5, 6)       # Shape of the primary example tensor
CP_RANK = 3                     # Rank for CP decomposition
TUCKER_RANKS = (2, 3, 3)        # Multilinear rank for Tucker decomposition
ALS_MAX_ITER = 200              # Maximum ALS iterations
ALS_TOL = 1e-8                  # ALS convergence tolerance

# Inertia tensor parameters
N_MASS_POINTS = 500             # Number of point masses for discretization
BODY_DENSITY = 1.0              # Uniform density

# Robot arm parameters
N_JOINTS = 3                    # Number of revolute joints
LINK_LENGTHS = np.array([1.0, 0.8, 0.5])  # Link lengths
N_CONFIG_SAMPLES = 50           # Configuration space samples

# Verification tolerances
RTOL = 1e-10                    # Relative tolerance for numerical checks
ATOL = 1e-10                    # Absolute tolerance for numerical checks

# Visualization
COLOR_PRIMARY = 'steelblue'
COLOR_SECONDARY = 'coral'
COLOR_TERTIARY = 'seagreen'
COLOR_ACCENT = 'goldenrod'

---
## 2. Tensor Fundamentals

### Terminology

An $N$th-order tensor $\mathcal{X} \in \mathbb{R}^{I_1 \times I_2 \times \cdots \times I_N}$ is characterized by:

- **Order** (number of modes/ways): $N$
- **Mode-$n$ dimension**: $I_n$ (size along the $n$-th axis)
- **Elements**: $x_{i_1 i_2 \cdots i_N}$ with $1 \leq i_n \leq I_n$

### Fibers and Slices

**Fibers** are the higher-order analogue of rows and columns:
- A **mode-$n$ fiber** is obtained by fixing all indices except the $n$-th: $\mathcal{X}(:, i_2, i_3)$ for mode-1 fibers

**Slices** are 2D sections obtained by fixing all but two indices:
- **Horizontal slices**: $\mathcal{X}(i_1, :, :)$
- **Lateral slices**: $\mathcal{X}(:, i_2, :)$
- **Frontal slices**: $\mathcal{X}(:, :, i_3)$

### Matricization (Unfolding)

The **mode-$n$ unfolding** (or matricization) of $\mathcal{X}$ rearranges the tensor into a matrix $\mathbf{X}_{(n)} \in \mathbb{R}^{I_n \times (I_1 \cdots I_{n-1} I_{n+1} \cdots I_N)}$ by mapping mode-$n$ fibers to columns:

$$\boxed{x_{i_1 i_2 \cdots i_N} \mapsto (\mathbf{X}_{(n)})_{i_n, j}}$$

where $j = 1 + \sum_{k \neq n} (i_k - 1) \prod_{m < k, m \neq n} I_m$.

Unfolding is the bridge between tensor and matrix operations --- it allows us to apply standard linear algebra tools mode-by-mode.

In [ ]:
class Tensor:
    """Multi-dimensional tensor with unfolding and mode-product operations.

    Wraps a NumPy ndarray and provides tensor-specific operations including
    matricization (unfolding), refolding, and n-mode products.

    Args:
        data: NumPy array of any shape. Shape: (I_1, I_2, ..., I_N).

    Attributes:
        data: The underlying NumPy array.
        shape: Tuple of mode dimensions.
        order: Number of modes (dimensions).
    """

    def __init__(self, data):
        self.data = np.array(data, dtype=float)
        self.shape = self.data.shape
        self.order = len(self.shape)

    def unfold(self, mode):
        """Compute the mode-n unfolding (matricization) of the tensor.

        Rearranges the tensor into a matrix where mode-n fibers become rows.
        Uses the convention from Kolda & Bader (2009).

        Args:
            mode: The mode along which to unfold (0-indexed). Integer.

        Returns:
            Matrix of shape (I_mode, prod(I_k for k != mode)).
            Shape: (I_mode, J) where J = product of all other dimensions.
        """
        n = mode
        # Move mode-n to the front, then flatten the rest
        # Ordering: mode n, then modes 0,1,...,n-1,n+1,...,N-1
        order = [n] + [i for i in range(self.order) if i != n]
        transposed = np.transpose(self.data, order)
        return transposed.reshape(self.shape[n], -1)

    @staticmethod
    def refold(matrix, mode, shape):
        """Refold a mode-n unfolded matrix back into a tensor.

        Inverse operation of unfold(mode).

        Args:
            matrix: Unfolded matrix. Shape: (I_mode, J).
            mode: The mode used for unfolding. Integer.
            shape: Original tensor shape. Tuple of ints.

        Returns:
            Tensor object with the specified shape.
        """
        N = len(shape)
        order = [mode] + [i for i in range(N) if i != mode]
        # Shape after transposition
        trans_shape = [shape[i] for i in order]
        # Reshape then inverse-transpose
        data = matrix.reshape(trans_shape)
        # Compute inverse permutation
        inv_order = [0] * N
        for i, o in enumerate(order):
            inv_order[o] = i
        data = np.transpose(data, inv_order)
        return Tensor(data)

    def mode_product(self, matrix, mode):
        """Compute the n-mode product of the tensor with a matrix.

        The n-mode product of tensor X with matrix U along mode n is:
            Y = X x_n U
        where Y_{i_1...i_{n-1} j i_{n+1}...i_N} = sum_i_n X_{i_1...i_N} U_{j,i_n}

        Args:
            matrix: Factor matrix. Shape: (J, I_mode).
            mode: The mode along which to multiply. Integer.

        Returns:
            Tensor with mode-n dimension changed from I_mode to J.
        """
        unfolded = self.unfold(mode)
        result_unfolded = matrix @ unfolded
        new_shape = list(self.shape)
        new_shape[mode] = matrix.shape[0]
        return Tensor.refold(result_unfolded, mode, tuple(new_shape))

    def norm(self):
        """Compute the Frobenius norm of the tensor.

        Returns:
            Frobenius norm (scalar).
        """
        return np.linalg.norm(self.data)

    def __repr__(self):
        return f"Tensor(shape={self.shape}, order={self.order})"

    def __sub__(self, other):
        return Tensor(self.data - other.data)

    def __add__(self, other):
        return Tensor(self.data + other.data)

In [ ]:
# ---- Demonstrate Tensor Basics ----
X = Tensor(np.random.randn(*TENSOR_SHAPE))
print(f"Tensor X: {X}")
print(f"Shape: {X.shape}")
print(f"Order: {X.order}")
print(f"Total elements: {np.prod(X.shape)}")
print(f"Frobenius norm: {X.norm():.4f}")
print()

# Unfoldings
for mode in range(X.order):
    unfolded = X.unfold(mode)
    print(f"Mode-{mode} unfolding: {X.shape} -> {unfolded.shape}")

In [ ]:
# ---- Verification: Unfold/Refold Roundtrip ----
X_test = Tensor(np.random.randn(*TENSOR_SHAPE))

all_pass = True
for mode in range(X_test.order):
    unfolded = X_test.unfold(mode)
    refolded = Tensor.refold(unfolded, mode, X_test.shape)
    rel_err = np.linalg.norm(refolded.data - X_test.data) / X_test.norm()
    status = "PASS" if rel_err < RTOL else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"Unfold/refold roundtrip mode-{mode}: max relative error = {rel_err:.2e} [{status}]")

print(f"\nAll unfold/refold roundtrip tests: [{'PASS' if all_pass else 'FAIL'}]")

In [ ]:
# ---- Visualize Tensor Slices and Fibers ----
X_small = Tensor(np.random.randn(4, 5, 6))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Frontal slices
slice_indices = [0, 2, 5]
for idx, (ax, si) in enumerate(zip(axes, slice_indices)):
    im = ax.imshow(X_small.data[:, :, si], cmap='RdBu_r', aspect='auto',
                   vmin=-2.5, vmax=2.5)
    ax.set_xlabel('Mode 1')
    ax.set_ylabel('Mode 0')
    ax.set_title(f'Frontal Slice $\\mathcal{{X}}(:,:,{si})$')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Frontal Slices of a 3rd-Order Tensor', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Visualize Unfoldings ----
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for mode in range(3):
    ax = axes[mode]
    unfolded = X_small.unfold(mode)
    im = ax.imshow(unfolded, cmap='RdBu_r', aspect='auto', vmin=-2.5, vmax=2.5)
    ax.set_xlabel('Column index')
    ax.set_ylabel('Row index')
    ax.set_title(f'Mode-{mode} Unfolding $\\mathbf{{X}}_{{({mode})}}$: {unfolded.shape}')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Matricizations of a (4, 5, 6) Tensor', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Tensor Products & Multilinear Maps

### Outer Product

The **outer product** of vectors $\mathbf{a} \in \mathbb{R}^{I}$, $\mathbf{b} \in \mathbb{R}^{J}$, $\mathbf{c} \in \mathbb{R}^{K}$ produces a rank-1 tensor:

$$\boxed{(\mathbf{a} \circ \mathbf{b} \circ \mathbf{c})_{ijk} = a_i \, b_j \, c_k}$$

This is the tensor analogue of the matrix outer product $\mathbf{a}\mathbf{b}^T$.

### Kronecker Product

The **Kronecker product** $\mathbf{A} \otimes \mathbf{B}$ of matrices is closely related to tensor products. For vectors, $\mathbf{a} \otimes \mathbf{b} = \text{vec}(\mathbf{b} \mathbf{a}^T)$. A key identity:

$$(\mathbf{A} \otimes \mathbf{B})(\mathbf{C} \otimes \mathbf{D}) = (\mathbf{AC}) \otimes (\mathbf{BD})$$

### Khatri-Rao Product

The **Khatri-Rao product** $\mathbf{A} \odot \mathbf{B}$ is the "column-wise Kronecker product":

$$\mathbf{A} \odot \mathbf{B} = [\mathbf{a}_1 \otimes \mathbf{b}_1 \; | \; \mathbf{a}_2 \otimes \mathbf{b}_2 \; | \; \cdots \; | \; \mathbf{a}_R \otimes \mathbf{b}_R]$$

This product is fundamental to the CP decomposition, as the mode-$n$ unfolding of a CP tensor can be expressed using Khatri-Rao products.

In [ ]:
def outer_product(*vectors):
    """Compute the outer product of multiple vectors to form a rank-1 tensor.

    Args:
        *vectors: Variable number of 1D arrays. Shape: (I_k,) for k=1,...,N.

    Returns:
        Tensor of shape (I_1, I_2, ..., I_N) representing the outer product.
    """
    result = vectors[0]
    for v in vectors[1:]:
        result = np.multiply.outer(result, v)
    return Tensor(result)


def khatri_rao(A, B):
    """Compute the Khatri-Rao (column-wise Kronecker) product.

    Args:
        A: First matrix. Shape: (I, R).
        B: Second matrix. Shape: (J, R).

    Returns:
        Khatri-Rao product. Shape: (I*J, R).
    """
    I, R = A.shape
    J = B.shape[0]
    assert A.shape[1] == B.shape[1], "Matrices must have the same number of columns"
    result = np.zeros((I * J, R))
    for r in range(R):
        result[:, r] = np.kron(A[:, r], B[:, r])
    return result


# ---- Demonstrate outer product ----
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0])
c = np.array([6.0, 7.0, 8.0, 9.0])

T_rank1 = outer_product(a, b, c)
print(f"Outer product a o b o c: shape = {T_rank1.shape}")
print(f"T[1,0,2] = a[1]*b[0]*c[2] = {a[1]}*{b[0]}*{c[2]} = {a[1]*b[0]*c[2]}")
print(f"Actual T[1,0,2] = {T_rank1.data[1,0,2]}")
print()

# Verify rank-1 property: every frontal slice is a scaled outer product of a, b
for k in range(len(c)):
    expected_slice = c[k] * np.outer(a, b)
    err = np.linalg.norm(T_rank1.data[:, :, k] - expected_slice)
    print(f"Frontal slice k={k}: error = {err:.2e}")

In [ ]:
# ---- Kronecker and Khatri-Rao Product Examples ----
A = np.array([[1, 2], [3, 4], [5, 6]])
B = np.array([[7, 8], [9, 10]])

# Kronecker product
kron_AB = np.kron(A, B)
print(f"A: {A.shape}, B: {B.shape}")
print(f"Kronecker A (x) B: {kron_AB.shape}")
print()

# Khatri-Rao product
kr_AB = khatri_rao(A, B)
print(f"Khatri-Rao A (*) B: {kr_AB.shape}")
print(f"Column 0: kron(A[:,0], B[:,0]) = kron({A[:,0]}, {B[:,0]}) = {np.kron(A[:,0], B[:,0])}")
print(f"Actual column 0: {kr_AB[:,0]}")

---
## 4. CP Decomposition

The **CANDECOMP/PARAFAC (CP) decomposition** approximates a tensor as a sum of rank-1 tensors:

$$\boxed{\mathcal{X} \approx \sum_{r=1}^{R} \lambda_r \, \mathbf{a}_r \circ \mathbf{b}_r \circ \mathbf{c}_r = [\![\boldsymbol{\lambda}; \mathbf{A}, \mathbf{B}, \mathbf{C}]\!]}$$

where $\mathbf{A} = [\mathbf{a}_1 | \cdots | \mathbf{a}_R] \in \mathbb{R}^{I \times R}$, and similarly for $\mathbf{B}$ and $\mathbf{C}$.

The smallest $R$ for which exact equality holds is the **tensor rank**.

### Mode-$n$ Unfolding of a CP Tensor

A key identity relates the CP decomposition to matrix factorizations:

$$\mathbf{X}_{(0)} = \mathbf{A} \, \text{diag}(\boldsymbol{\lambda}) \, (\mathbf{C} \odot \mathbf{B})^T$$

### Alternating Least Squares (ALS)

ALS minimizes $\|\mathcal{X} - [\![\mathbf{A}, \mathbf{B}, \mathbf{C}]\!]\|_F^2$ by alternately fixing two factor matrices and solving for the third:

1. Fix $\mathbf{B}, \mathbf{C}$; solve $\mathbf{X}_{(0)} \approx \mathbf{A} (\mathbf{C} \odot \mathbf{B})^T$ for $\mathbf{A}$
2. Fix $\mathbf{A}, \mathbf{C}$; solve $\mathbf{X}_{(1)} \approx \mathbf{B} (\mathbf{C} \odot \mathbf{A})^T$ for $\mathbf{B}$
3. Fix $\mathbf{A}, \mathbf{B}$; solve $\mathbf{X}_{(2)} \approx \mathbf{C} (\mathbf{B} \odot \mathbf{A})^T$ for $\mathbf{C}$

Each step is a standard least-squares problem.

In [ ]:
def cp_reconstruct(factors, weights=None):
    """Reconstruct a tensor from its CP decomposition.

    Args:
        factors: List of factor matrices [A, B, C, ...]. 
                 Shape: [(I_1, R), (I_2, R), ..., (I_N, R)].
        weights: Optional weight vector. Shape: (R,). If None, all ones.

    Returns:
        Tensor reconstructed from the CP decomposition.
    """
    R = factors[0].shape[1]
    if weights is None:
        weights = np.ones(R)
    shape = tuple(f.shape[0] for f in factors)
    result = np.zeros(shape)
    for r in range(R):
        component = weights[r] * outer_product(*[f[:, r] for f in factors]).data
        result += component
    return Tensor(result)


def cp_als(X, rank, max_iter=ALS_MAX_ITER, tol=ALS_TOL):
    """Compute CP decomposition using Alternating Least Squares.

    Minimizes ||X - [[A, B, C, ...]]||_F^2 by alternately solving
    for each factor matrix while holding the others fixed.

    Args:
        X: Input tensor. Tensor object.
        rank: Number of rank-1 components. Integer.
        max_iter: Maximum ALS iterations. Integer.
        tol: Convergence tolerance on relative change. Scalar.

    Returns:
        factors: List of factor matrices. Shape: [(I_n, R) for each mode].
        weights: Normalization weights. Shape: (R,).
        errors: List of relative reconstruction errors per iteration.
    """
    N = X.order
    # Initialize factor matrices randomly
    factors = [np.random.randn(X.shape[n], rank) for n in range(N)]

    errors = []
    X_norm = X.norm()

    for iteration in range(max_iter):
        for n in range(N):
            # Compute the Khatri-Rao product of all factors except mode n
            # Order: last mode to first mode, skipping mode n
            kr_indices = [i for i in range(N) if i != n]
            # Start from the last index
            kr = factors[kr_indices[-1]]
            for i in reversed(kr_indices[:-1]):
                kr = khatri_rao(factors[i], kr)

            # Solve X_(n) = A_n * kr^T  =>  A_n = X_(n) * kr * (kr^T kr)^{-1}
            X_n = X.unfold(n)
            V = kr.T @ kr  # Shape: (R, R) -- Hadamard product of Grammians
            factors[n] = X_n @ kr @ np.linalg.pinv(V)

        # Normalize columns of factor matrices
        weights = np.ones(rank)
        for n in range(N):
            col_norms = np.linalg.norm(factors[n], axis=0)
            col_norms = np.maximum(col_norms, 1e-15)
            weights *= col_norms
            factors[n] = factors[n] / col_norms[np.newaxis, :]

        # Compute reconstruction error
        X_approx = cp_reconstruct(factors, weights)
        rel_error = (X - X_approx).norm() / X_norm
        errors.append(rel_error)

        # Check convergence
        if iteration > 0 and abs(errors[-1] - errors[-2]) < tol:
            break

    return factors, weights, errors

In [ ]:
# ---- Create a test tensor with known CP structure ----
# Build a rank-3 tensor by summing 3 rank-1 components
np.random.seed(42)
true_factors = [
    np.random.randn(TENSOR_SHAPE[0], CP_RANK),
    np.random.randn(TENSOR_SHAPE[1], CP_RANK),
    np.random.randn(TENSOR_SHAPE[2], CP_RANK),
]
true_weights = np.array([5.0, 3.0, 1.0])

X_cp = cp_reconstruct(true_factors, true_weights)
# Add small noise
noise_level = 0.01
X_noisy = Tensor(X_cp.data + noise_level * np.random.randn(*TENSOR_SHAPE))

print(f"True tensor norm:  {X_cp.norm():.4f}")
print(f"Noise level:       {noise_level}")
print(f"Noisy tensor norm: {X_noisy.norm():.4f}")
print(f"SNR:               {20 * np.log10(X_cp.norm() / (noise_level * np.linalg.norm(np.random.randn(*TENSOR_SHAPE)))):.1f} dB")

In [ ]:
# ---- Run CP-ALS ----
np.random.seed(42)
cp_factors, cp_weights, cp_errors = cp_als(X_noisy, CP_RANK, max_iter=ALS_MAX_ITER)

print(f"CP-ALS converged in {len(cp_errors)} iterations")
print(f"Final relative error: {cp_errors[-1]:.6e}")
print(f"Weights: {cp_weights}")
print()

# Verify error decreases monotonically
is_decreasing = all(cp_errors[i] >= cp_errors[i+1] - 1e-15 for i in range(len(cp_errors) - 1))
status = "PASS" if is_decreasing else "FAIL"
print(f"CP reconstruction error decreases monotonically: [{status}]")

In [ ]:
# ---- CP Decomposition: Error vs Rank ----
np.random.seed(42)
ranks_to_test = [1, 2, 3, 4, 5, 6, 8, 10]
cp_final_errors = []

for r in ranks_to_test:
    _, _, errs = cp_als(X_noisy, r, max_iter=ALS_MAX_ITER)
    cp_final_errors.append(errs[-1])
    print(f"Rank {r:2d}: relative error = {errs[-1]:.6e}")

print()
# Verify: error at true rank (3) should be close to noise level
noise_ratio = noise_level * np.sqrt(np.prod(TENSOR_SHAPE)) / X_cp.norm()
status = "PASS" if cp_final_errors[2] < 2 * noise_ratio else "FAIL"
print(f"CP error at true rank ~ noise floor: {cp_final_errors[2]:.4e} vs {noise_ratio:.4e} [{status}]")

---
## 5. Tucker Decomposition

The **Tucker decomposition** expresses a tensor as a core tensor multiplied by factor matrices along each mode:

$$\boxed{\mathcal{X} \approx \mathcal{G} \times_1 \mathbf{U}_1 \times_2 \mathbf{U}_2 \times_3 \mathbf{U}_3}$$

where $\mathcal{G} \in \mathbb{R}^{R_1 \times R_2 \times R_3}$ is the **core tensor** and $\mathbf{U}_n \in \mathbb{R}^{I_n \times R_n}$ are orthogonal factor matrices.

The tuple $(R_1, R_2, R_3)$ is called the **multilinear rank**.

### Higher-Order SVD (HOSVD)

The HOSVD computes the Tucker decomposition by performing SVD on each mode unfolding independently:

1. For each mode $n$: compute SVD of $\mathbf{X}_{(n)} = \mathbf{U}_n \boldsymbol{\Sigma}_n \mathbf{V}_n^T$ and keep the first $R_n$ left singular vectors
2. Form the core tensor: $\mathcal{G} = \mathcal{X} \times_1 \mathbf{U}_1^T \times_2 \mathbf{U}_2^T \times_3 \mathbf{U}_3^T$

Unlike the matrix SVD, HOSVD does not yield the optimal rank-$(R_1, R_2, R_3)$ approximation, but it provides a good initialization and has useful orthogonality properties.

In [ ]:
def hosvd(X, ranks):
    """Compute Higher-Order SVD (Tucker decomposition via truncated SVDs).

    For each mode n, computes the SVD of the mode-n unfolding and retains
    the leading R_n left singular vectors. The core tensor is then computed
    by projecting X onto the factor matrices.

    Args:
        X: Input tensor. Tensor object.
        ranks: Tuple of target ranks (R_1, R_2, ..., R_N). Tuple of ints.

    Returns:
        core: Core tensor G. Tensor object with shape = ranks.
        factors: List of orthogonal factor matrices [U_1, ..., U_N].
                 Shape: [(I_n, R_n) for each mode].
    """
    N = X.order
    factors = []

    for n in range(N):
        X_n = X.unfold(n)
        U, S, Vt = np.linalg.svd(X_n, full_matrices=False)
        factors.append(U[:, :ranks[n]])

    # Compute core tensor: G = X x_1 U_1^T x_2 U_2^T ... x_N U_N^T
    core = X
    for n in range(N):
        core = core.mode_product(factors[n].T, n)

    return core, factors


def tucker_reconstruct(core, factors):
    """Reconstruct a tensor from its Tucker decomposition.

    Args:
        core: Core tensor. Tensor object.
        factors: List of factor matrices [U_1, ..., U_N].
                 Shape: [(I_n, R_n) for each mode].

    Returns:
        Reconstructed tensor. Tensor object.
    """
    result = core
    for n in range(len(factors)):
        result = result.mode_product(factors[n], n)
    return result

In [ ]:
# ---- Run HOSVD ----
core, tucker_factors = hosvd(X_noisy, TUCKER_RANKS)

print(f"Input tensor shape:  {X_noisy.shape}")
print(f"Tucker ranks:        {TUCKER_RANKS}")
print(f"Core tensor shape:   {core.shape}")
print(f"Core tensor norm:    {core.norm():.4f}")
print()

for n, U in enumerate(tucker_factors):
    print(f"Factor U_{n}: {U.shape}")

# Reconstruct
X_tucker = tucker_reconstruct(core, tucker_factors)
tucker_rel_error = (X_noisy - X_tucker).norm() / X_noisy.norm()
print(f"\nTucker relative reconstruction error: {tucker_rel_error:.6e}")

In [ ]:
# ---- Verification: HOSVD Orthogonality ----
all_pass = True
for n, U in enumerate(tucker_factors):
    gram = U.T @ U
    identity = np.eye(gram.shape[0])
    orth_error = np.linalg.norm(gram - identity)
    status = "PASS" if orth_error < 1e-10 else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"HOSVD orthogonality U_{n}^T U_{n} = I: max relative error = {orth_error:.2e} [{status}]")

print(f"\nAll HOSVD orthogonality tests: [{'PASS' if all_pass else 'FAIL'}]")

In [ ]:
# ---- Tucker: Error vs Rank ----
tucker_rank_configs = [
    (1, 1, 1), (1, 2, 2), (2, 2, 2), (2, 3, 3),
    (3, 3, 3), (3, 4, 4), (4, 5, 5), (4, 5, 6),
]
tucker_errors = []
tucker_compression = []

for ranks in tucker_rank_configs:
    c, fs = hosvd(X_noisy, ranks)
    X_rec = tucker_reconstruct(c, fs)
    rel_err = (X_noisy - X_rec).norm() / X_noisy.norm()
    tucker_errors.append(rel_err)
    # Compression ratio: original elements / (core + factor) elements
    n_core = np.prod(ranks)
    n_factors = sum(X_noisy.shape[n] * ranks[n] for n in range(X_noisy.order))
    ratio = np.prod(X_noisy.shape) / (n_core + n_factors)
    tucker_compression.append(ratio)
    print(f"Ranks {str(ranks):12s}: error = {rel_err:.6e}, compression = {ratio:.2f}x")

# Verify error decreases with rank
is_decreasing = all(tucker_errors[i] >= tucker_errors[i+1] - 1e-12 for i in range(len(tucker_errors) - 1))
status = "PASS" if is_decreasing else "FAIL"
print(f"\nTucker reconstruction error decreases with rank: [{status}]")

---
## 6. Tensor Contraction & Einstein Notation

### Index Notation

In index notation, tensor operations are expressed by specifying how indices are combined. The **Einstein summation convention** implies summation over repeated indices:

$$C_{ik} = A_{ij} B_{jk} = \sum_j A_{ij} B_{jk}$$

### Tensor Contraction

**Contraction** generalizes matrix multiplication to tensors. Given tensors $\mathcal{A}$ and $\mathcal{B}$, contraction over mode $p$ of $\mathcal{A}$ and mode $q$ of $\mathcal{B}$ sums over the shared index:

$$\boxed{\mathcal{C}_{\ldots} = \sum_{k} \mathcal{A}_{\ldots k \ldots} \, \mathcal{B}_{\ldots k \ldots}}$$

Special cases:
- Matrix-matrix multiplication: $C_{ij} = \sum_k A_{ik} B_{kj}$
- Inner product: $s = \sum_{ij} A_{ij} B_{ij}$
- Mode-$n$ product: $Y_{i_1 \ldots j \ldots i_N} = \sum_{i_n} X_{i_1 \ldots i_n \ldots i_N} U_{j i_n}$

In [ ]:
def tensor_contract(A, B, axes_A, axes_B):
    """Contract two tensors over specified axes.

    Generalizes matrix multiplication to tensors. Sums over the
    specified axes of A and B, producing a tensor whose remaining
    axes are the non-contracted axes of A followed by those of B.

    Args:
        A: First tensor (numpy array). Shape: (I_1, ..., I_M).
        B: Second tensor (numpy array). Shape: (J_1, ..., J_P).
        axes_A: Axes of A to contract over. Int or list of ints.
        axes_B: Axes of B to contract over. Int or list of ints.

    Returns:
        Contracted tensor as numpy array.
    """
    # Ensure axes are lists
    if isinstance(axes_A, int):
        axes_A = [axes_A]
    if isinstance(axes_B, int):
        axes_B = [axes_B]

    # Verify dimensions match
    for a, b in zip(axes_A, axes_B):
        assert A.shape[a] == B.shape[b], (
            f"Dimension mismatch: A axis {a} has size {A.shape[a]}, "
            f"B axis {b} has size {B.shape[b]}"
        )

    # Determine free (non-contracted) axes
    free_A = [i for i in range(A.ndim) if i not in axes_A]
    free_B = [i for i in range(B.ndim) if i not in axes_B]

    # Move contracted axes to end (A) and beginning (B)
    A_transposed = np.transpose(A, free_A + axes_A)
    B_transposed = np.transpose(B, axes_B + free_B)

    # Reshape to 2D matrices
    shape_free_A = [A.shape[i] for i in free_A]
    shape_free_B = [B.shape[i] for i in free_B]
    shape_contracted = [A.shape[i] for i in axes_A]

    n_free_A = int(np.prod(shape_free_A)) if shape_free_A else 1
    n_free_B = int(np.prod(shape_free_B)) if shape_free_B else 1
    n_contracted = int(np.prod(shape_contracted))

    A_mat = A_transposed.reshape(n_free_A, n_contracted)
    B_mat = B_transposed.reshape(n_contracted, n_free_B)

    # Matrix multiply
    C_mat = A_mat @ B_mat

    # Reshape back
    result_shape = shape_free_A + shape_free_B
    if result_shape:
        return C_mat.reshape(result_shape)
    else:
        return C_mat.item()  # scalar case

In [ ]:
# ---- Verification: Contraction matches einsum ----
np.random.seed(42)
all_pass = True

# Test 1: Matrix multiplication  C_ij = A_ik B_kj
A_mat = np.random.randn(3, 4)
B_mat = np.random.randn(4, 5)
C_contract = tensor_contract(A_mat, B_mat, 1, 0)
C_einsum = np.einsum('ik,kj->ij', A_mat, B_mat)
err1 = np.linalg.norm(C_contract - C_einsum) / np.linalg.norm(C_einsum)
s1 = "PASS" if err1 < RTOL else "FAIL"
if s1 == "FAIL": all_pass = False
print(f"Matrix multiply contraction: max relative error = {err1:.2e} [{s1}]")

# Test 2: Tensor-matrix contraction  Y_ijk = X_ijl M_lk
X_t = np.random.randn(3, 4, 5)
M_t = np.random.randn(5, 6)
Y_contract = tensor_contract(X_t, M_t, 2, 0)
Y_einsum = np.einsum('ijl,lk->ijk', X_t, M_t)
err2 = np.linalg.norm(Y_contract - Y_einsum) / np.linalg.norm(Y_einsum)
s2 = "PASS" if err2 < RTOL else "FAIL"
if s2 == "FAIL": all_pass = False
print(f"Tensor-matrix contraction: max relative error = {err2:.2e} [{s2}]")

# Test 3: Double contraction (inner product)  s = A_ij B_ij
A_d = np.random.randn(3, 4)
B_d = np.random.randn(3, 4)
s_contract = tensor_contract(A_d, B_d, [0, 1], [0, 1])
s_einsum = np.einsum('ij,ij->', A_d, B_d)
err3 = abs(s_contract - s_einsum) / abs(s_einsum)
s3 = "PASS" if err3 < RTOL else "FAIL"
if s3 == "FAIL": all_pass = False
print(f"Double contraction (inner product): max relative error = {err3:.2e} [{s3}]")

# Test 4: Tensor-tensor contraction  C_ikl = A_ij B_jkl
A_tt = np.random.randn(3, 4)
B_tt = np.random.randn(4, 5, 6)
C_contract = tensor_contract(A_tt, B_tt, 1, 0)
C_einsum = np.einsum('ij,jkl->ikl', A_tt, B_tt)
err4 = np.linalg.norm(C_contract - C_einsum) / np.linalg.norm(C_einsum)
s4 = "PASS" if err4 < RTOL else "FAIL"
if s4 == "FAIL": all_pass = False
print(f"Tensor-tensor contraction: max relative error = {err4:.2e} [{s4}]")

# Test 5: Higher-order contraction  C_il = A_ijk B_jkl
A_ho = np.random.randn(3, 4, 5)
B_ho = np.random.randn(4, 5, 6)
C_contract = tensor_contract(A_ho, B_ho, [1, 2], [0, 1])
C_einsum = np.einsum('ijk,jkl->il', A_ho, B_ho)
err5 = np.linalg.norm(C_contract - C_einsum) / np.linalg.norm(C_einsum)
s5 = "PASS" if err5 < RTOL else "FAIL"
if s5 == "FAIL": all_pass = False
print(f"Higher-order contraction: max relative error = {err5:.2e} [{s5}]")

print(f"\nAll contraction tests: [{'PASS' if all_pass else 'FAIL'}]")

---
## 7. Application: Inertia Tensor

The **inertia tensor** of a rigid body describes its resistance to angular acceleration. For a body with mass density $\rho(\mathbf{r})$:

$$\boxed{I_{ij} = \int_V \rho(\mathbf{r}) \left( |\mathbf{r}|^2 \delta_{ij} - r_i r_j \right) dV}$$

where $\delta_{ij}$ is the Kronecker delta. For a discrete collection of point masses:

$$I_{ij} = \sum_k m_k \left( |\mathbf{r}_k|^2 \delta_{ij} - (r_k)_i (r_k)_j \right)$$

### Properties

1. **Symmetric**: $I_{ij} = I_{ji}$ (it is a symmetric 2nd-order tensor)
2. **Positive semi-definite**: eigenvalues $\geq 0$
3. **Principal axes**: eigenvectors of $\mathbf{I}$ define the body's principal axes of rotation

### Parallel Axis Theorem

The inertia tensor about a point displaced by $\mathbf{d}$ from the center of mass:

$$\boxed{I'_{ij} = I_{ij}^{\text{cm}} + M \left( |\mathbf{d}|^2 \delta_{ij} - d_i d_j \right)}$$

where $M$ is the total mass and $I^{\text{cm}}$ is the inertia about the center of mass.

In [ ]:
def inertia_tensor(positions, masses):
    """Compute the inertia tensor from point masses.

    Args:
        positions: Position vectors of point masses. Shape: (N, 3).
        masses: Mass of each point. Shape: (N,).

    Returns:
        I: 3x3 inertia tensor. Shape: (3, 3).
    """
    N = len(masses)
    I = np.zeros((3, 3))
    for k in range(N):
        r = positions[k]
        r_sq = np.dot(r, r)
        I += masses[k] * (r_sq * np.eye(3) - np.outer(r, r))
    return I


def parallel_axis_theorem(I_cm, total_mass, displacement):
    """Apply the parallel axis theorem to translate the inertia tensor.

    Args:
        I_cm: Inertia tensor about center of mass. Shape: (3, 3).
        total_mass: Total mass of the body. Scalar.
        displacement: Translation vector from CM to new origin. Shape: (3,).

    Returns:
        I_new: Inertia tensor about the displaced origin. Shape: (3, 3).
    """
    d = displacement
    d_sq = np.dot(d, d)
    return I_cm + total_mass * (d_sq * np.eye(3) - np.outer(d, d))

In [ ]:
# ---- Generate a solid ellipsoid via rejection sampling ----
np.random.seed(42)
SEMI_AXES = np.array([2.0, 1.0, 0.5])  # Semi-axes a, b, c

# Sample uniformly within the ellipsoid
points = []
while len(points) < N_MASS_POINTS:
    candidate = np.random.uniform(-1, 1, 3) * SEMI_AXES
    if np.sum((candidate / SEMI_AXES)**2) <= 1.0:
        points.append(candidate)

positions = np.array(points)
# Equal mass for each point (uniform density)
total_mass = BODY_DENSITY * (4.0/3.0) * np.pi * np.prod(SEMI_AXES)
masses = np.full(N_MASS_POINTS, total_mass / N_MASS_POINTS)

# Shift to center of mass
cm = np.average(positions, weights=masses, axis=0)
positions_cm = positions - cm

print(f"Ellipsoid semi-axes: {SEMI_AXES}")
print(f"Total mass: {total_mass:.4f}")
print(f"Number of point masses: {N_MASS_POINTS}")
print(f"Center of mass (should be ~0): {cm}")

In [ ]:
# ---- Compute Inertia Tensor ----
I_cm = inertia_tensor(positions_cm, masses)

# Analytical inertia tensor for a uniform ellipsoid (about CM)
a, b, c = SEMI_AXES
M = total_mass
I_analytical = np.diag([
    M / 5.0 * (b**2 + c**2),
    M / 5.0 * (a**2 + c**2),
    M / 5.0 * (a**2 + b**2),
])

print("Numerical inertia tensor (about CM):")
print(np.array2string(I_cm, precision=4, suppress_small=True))
print()
print("Analytical inertia tensor (about CM):")
print(np.array2string(I_analytical, precision=4, suppress_small=True))
print()

# Principal moments (eigenvalues)
eigenvalues, eigenvectors = np.linalg.eigh(I_cm)
print("Principal moments of inertia:")
for i in range(3):
    print(f"  I_{i+1} = {eigenvalues[i]:.4f} (analytical: {np.sort(np.diag(I_analytical))[i]:.4f})")

In [ ]:
# ---- Verification: Inertia Tensor Symmetry + Parallel Axis Theorem ----

# Test 1: Symmetry
sym_error = np.linalg.norm(I_cm - I_cm.T) / np.linalg.norm(I_cm)
status_sym = "PASS" if sym_error < RTOL else "FAIL"
print(f"Inertia tensor symmetry: max relative error = {sym_error:.2e} [{status_sym}]")

# Test 2: Positive semi-definiteness
min_eigenvalue = np.min(eigenvalues)
status_psd = "PASS" if min_eigenvalue >= -ATOL else "FAIL"
print(f"Positive semi-definite (min eigenvalue = {min_eigenvalue:.4e}): [{status_psd}]")

# Test 3: Parallel axis theorem
d = np.array([1.0, 2.0, 0.5])  # Displacement vector

# Method 1: Compute directly about displaced origin
positions_displaced = positions_cm - d  # positions relative to displaced origin
I_direct = inertia_tensor(positions_displaced, masses)

# Method 2: Use parallel axis theorem
I_pat = parallel_axis_theorem(I_cm, np.sum(masses), d)

pat_error = np.linalg.norm(I_direct - I_pat) / np.linalg.norm(I_direct)
status_pat = "PASS" if pat_error < 1e-10 else "FAIL"
print(f"Parallel axis theorem: max relative error = {pat_error:.2e} [{status_pat}]")

---
## 8. Application: Manipulability Tensor

For a robot arm with Jacobian $\mathbf{J}(\mathbf{q}) \in \mathbb{R}^{m \times n}$, the **Yoshikawa manipulability** is a scalar measure of dexterity:

$$w(\mathbf{q}) = \sqrt{\det(\mathbf{J}\mathbf{J}^T)}$$

This tells us *how much* the robot can move, but not *in which directions*. The **manipulability ellipsoid** defined by $\mathbf{J}\mathbf{J}^T$ provides directional information.

### Higher-Order Manipulability

We can extend this to a **manipulability tensor** that captures how the manipulability structure varies across configuration space:

$$\mathcal{M}_{ijk} = (\mathbf{J}\mathbf{J}^T)_{ij} \big|_{\mathbf{q}_k}$$

This 3rd-order tensor $(m \times m \times K)$ stacks the manipulability matrices evaluated at $K$ configurations. Applying Tucker decomposition reveals the **dominant manipulability patterns** and their variation across the workspace.

In [ ]:
def forward_kinematics_2d(q, link_lengths):
    """Compute end-effector position for a planar revolute arm.

    Args:
        q: Joint angles. Shape: (N_JOINTS,).
        link_lengths: Link lengths. Shape: (N_JOINTS,).

    Returns:
        pos: End-effector position [x, y]. Shape: (2,).
    """
    cumulative_angle = np.cumsum(q)
    x = np.sum(link_lengths * np.cos(cumulative_angle))
    y = np.sum(link_lengths * np.sin(cumulative_angle))
    return np.array([x, y])


def jacobian_2d(q, link_lengths):
    """Compute the geometric Jacobian for a planar revolute arm.

    Args:
        q: Joint angles. Shape: (N_JOINTS,).
        link_lengths: Link lengths. Shape: (N_JOINTS,).

    Returns:
        J: Jacobian matrix. Shape: (2, N_JOINTS).
    """
    n = len(q)
    J = np.zeros((2, n))
    for j in range(n):
        # Contribution of joint j to end-effector velocity
        for k in range(j, n):
            angle = np.sum(q[:k+1])
            J[0, j] += -link_lengths[k] * np.sin(angle)
            J[1, j] += link_lengths[k] * np.cos(angle)
    return J


def manipulability_tensor(configurations, link_lengths):
    """Compute the manipulability tensor across configurations.

    Stacks the manipulability matrices J*J^T evaluated at each configuration
    into a 3rd-order tensor.

    Args:
        configurations: Joint angle configurations. Shape: (K, N_JOINTS).
        link_lengths: Link lengths. Shape: (N_JOINTS,).

    Returns:
        M_tensor: Manipulability tensor. Tensor object with shape (2, 2, K).
        manipulabilities: Scalar manipulability at each config. Shape: (K,).
    """
    K = configurations.shape[0]
    m = 2  # task space dimension
    M = np.zeros((m, m, K))
    w = np.zeros(K)

    for k in range(K):
        J = jacobian_2d(configurations[k], link_lengths)
        JJT = J @ J.T
        M[:, :, k] = JJT
        w[k] = np.sqrt(max(np.linalg.det(JJT), 0.0))

    return Tensor(M), w

In [ ]:
# ---- Generate configurations along a trajectory ----
np.random.seed(42)
t_param = np.linspace(0, 2 * np.pi, N_CONFIG_SAMPLES)

# Smooth trajectory through joint space
configs = np.column_stack([
    0.5 * np.sin(t_param) + 0.3,
    0.8 * np.cos(0.7 * t_param) + 0.5,
    0.3 * np.sin(1.3 * t_param) - 0.2,
])

M_tensor, manip_scalar = manipulability_tensor(configs, LINK_LENGTHS)

print(f"Manipulability tensor shape: {M_tensor.shape}")
print(f"Scalar manipulability range: [{manip_scalar.min():.4f}, {manip_scalar.max():.4f}]")
print(f"Mean manipulability: {manip_scalar.mean():.4f}")

In [ ]:
# ---- Tucker decomposition of the manipulability tensor ----
manip_ranks = (2, 2, 5)
core_m, factors_m = hosvd(M_tensor, manip_ranks)

M_rec = tucker_reconstruct(core_m, factors_m)
manip_rel_error = (M_tensor - M_rec).norm() / M_tensor.norm()

print(f"Tucker ranks for manipulability tensor: {manip_ranks}")
print(f"Core tensor shape: {core_m.shape}")
print(f"Relative reconstruction error: {manip_rel_error:.6e}")
print(f"Configuration factor U_3 captures {factors_m[2].shape[1]} dominant patterns")

In [ ]:
# ---- Robot Arm Visualization with Manipulability ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Arm configurations colored by manipulability
ax = axes[0]
n_show = 10
show_indices = np.linspace(0, N_CONFIG_SAMPLES - 1, n_show, dtype=int)
colors = plt.cm.viridis(np.linspace(0, 1, n_show))

for idx, si in enumerate(show_indices):
    q = configs[si]
    # Compute joint positions
    x_joints = [0.0]
    y_joints = [0.0]
    cumangle = 0.0
    for j in range(N_JOINTS):
        cumangle += q[j]
        x_joints.append(x_joints[-1] + LINK_LENGTHS[j] * np.cos(cumangle))
        y_joints.append(y_joints[-1] + LINK_LENGTHS[j] * np.sin(cumangle))
    ax.plot(x_joints, y_joints, 'o-', color=colors[idx], linewidth=2, markersize=4, alpha=0.7)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Robot Arm Configurations')
ax.set_aspect('equal')

# Panel 2: Scalar manipulability along trajectory
ax = axes[1]
ax.plot(t_param, manip_scalar, color=COLOR_PRIMARY, linewidth=2)
ax.fill_between(t_param, 0, manip_scalar, alpha=0.2, color=COLOR_PRIMARY)
ax.set_xlabel('Trajectory Parameter $t$')
ax.set_ylabel('Manipulability $w(\\mathbf{q})$')
ax.set_title('Scalar Manipulability Along Trajectory')

# Panel 3: Configuration-space factors from Tucker decomposition
ax = axes[2]
for r in range(factors_m[2].shape[1]):
    ax.plot(t_param, factors_m[2][:, r], linewidth=2, label=f'Mode {r+1}')
ax.set_xlabel('Trajectory Parameter $t$')
ax.set_ylabel('Factor Weight')
ax.set_title('Tucker Configuration Factors $\\mathbf{U}_3$')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Manipulability Ellipses at Selected Configurations ----
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

n_ellipses = 8
ellipse_indices = np.linspace(0, N_CONFIG_SAMPLES - 1, n_ellipses, dtype=int)
colors_e = plt.cm.viridis(np.linspace(0, 1, n_ellipses))
scale = 0.15  # Scale for ellipse visualization

for idx, si in enumerate(ellipse_indices):
    q = configs[si]
    ee = forward_kinematics_2d(q, LINK_LENGTHS)
    
    # Draw arm
    x_joints = [0.0]
    y_joints = [0.0]
    cumangle = 0.0
    for j in range(N_JOINTS):
        cumangle += q[j]
        x_joints.append(x_joints[-1] + LINK_LENGTHS[j] * np.cos(cumangle))
        y_joints.append(y_joints[-1] + LINK_LENGTHS[j] * np.sin(cumangle))
    ax.plot(x_joints, y_joints, 'o-', color=colors_e[idx], linewidth=1.5,
            markersize=3, alpha=0.5)

    # Draw manipulability ellipse
    JJT = M_tensor.data[:, :, si]
    eigvals, eigvecs = np.linalg.eigh(JJT)
    eigvals = np.maximum(eigvals, 0)
    theta_e = np.linspace(0, 2 * np.pi, 100)
    ellipse = scale * eigvecs @ np.diag(np.sqrt(eigvals)) @ np.array([np.cos(theta_e), np.sin(theta_e)])
    ax.fill(ee[0] + ellipse[0], ee[1] + ellipse[1], alpha=0.3, color=colors_e[idx])
    ax.plot(ee[0] + ellipse[0], ee[1] + ellipse[1], color=colors_e[idx], linewidth=1.5)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Manipulability Ellipses at Selected Configurations')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 9. Visualizations

Comprehensive visualizations of tensor structure and decomposition quality.

In [ ]:
# ---- Tensor Slice Heatmaps ----
X_vis = X_noisy

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Top row: original tensor frontal slices
for k in range(3):
    ax = axes[0, k]
    im = ax.imshow(X_vis.data[:, :, k], cmap='RdBu_r', aspect='auto', vmin=-5, vmax=5)
    ax.set_xlabel('Mode 1')
    ax.set_ylabel('Mode 0')
    ax.set_title(f'Original $\\mathcal{{X}}(:,:,{k})$')
    plt.colorbar(im, ax=ax, shrink=0.8)

# Bottom row: CP-reconstructed frontal slices
X_cp_rec = cp_reconstruct(cp_factors, cp_weights)
for k in range(3):
    ax = axes[1, k]
    im = ax.imshow(X_cp_rec.data[:, :, k], cmap='RdBu_r', aspect='auto', vmin=-5, vmax=5)
    ax.set_xlabel('Mode 1')
    ax.set_ylabel('Mode 0')
    ax.set_title(f'CP Rank-{CP_RANK} $\\hat{{\\mathcal{{X}}}}(:,:,{k})$')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Original vs CP-Reconstructed Tensor Slices', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- CP Factor Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
mode_labels = ['Mode 0', 'Mode 1', 'Mode 2']
colors_cp = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_TERTIARY]

for n in range(3):
    ax = axes[n]
    for r in range(CP_RANK):
        ax.plot(cp_factors[n][:, r], 'o-', color=colors_cp[r], linewidth=2,
                markersize=5, label=f'Component {r+1} ($\\lambda={cp_weights[r]:.2f}$)')
    ax.set_xlabel(f'{mode_labels[n]} index')
    ax.set_ylabel('Factor value')
    ax.set_title(f'CP Factors: {mode_labels[n]}')
    ax.legend(fontsize=9)

plt.suptitle(f'CP Decomposition Factor Matrices (Rank {CP_RANK})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Tucker Core Structure ----
fig, axes = plt.subplots(1, TUCKER_RANKS[2], figsize=(4 * TUCKER_RANKS[2], 3.5))
if TUCKER_RANKS[2] == 1:
    axes = [axes]

for k in range(TUCKER_RANKS[2]):
    ax = axes[k]
    im = ax.imshow(core.data[:, :, k], cmap='RdBu_r', aspect='auto')
    ax.set_xlabel('Core Mode 1')
    ax.set_ylabel('Core Mode 0')
    ax.set_title(f'Core Slice $\\mathcal{{G}}(:,:,{k})$')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f'Tucker Core Tensor (shape {TUCKER_RANKS})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Inertia Ellipsoid (3D) ----
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot point masses (subsample for clarity)
subsample = np.random.choice(N_MASS_POINTS, min(100, N_MASS_POINTS), replace=False)
ax.scatter(positions_cm[subsample, 0], positions_cm[subsample, 1],
           positions_cm[subsample, 2], s=5, alpha=0.3, c=COLOR_PRIMARY)

# Compute and plot inertia ellipsoid
# The inertia ellipsoid has axes proportional to 1/sqrt(I_principal)
eigvals_inertia, eigvecs_inertia = np.linalg.eigh(I_cm)
radii = 1.0 / np.sqrt(eigvals_inertia / total_mass)  # Normalized radii

# Parametric ellipsoid surface
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 30)
x_e = radii[0] * np.outer(np.cos(u), np.sin(v))
y_e = radii[1] * np.outer(np.sin(u), np.sin(v))
z_e = radii[2] * np.outer(np.ones_like(u), np.cos(v))

# Rotate ellipsoid to align with principal axes
for i in range(len(u)):
    for j in range(len(v)):
        point = eigvecs_inertia @ np.array([x_e[i, j], y_e[i, j], z_e[i, j]])
        x_e[i, j], y_e[i, j], z_e[i, j] = point

ax.plot_surface(x_e, y_e, z_e, alpha=0.2, color=COLOR_SECONDARY)

# Plot principal axes
axis_scale = 2.5
colors_axes = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_TERTIARY]
labels = ['$e_1$', '$e_2$', '$e_3$']
for i in range(3):
    direction = eigvecs_inertia[:, i] * axis_scale
    ax.quiver(0, 0, 0, direction[0], direction[1], direction[2],
              color=colors_axes[i], linewidth=2.5, arrow_length_ratio=0.1)
    ax.text(direction[0] * 1.15, direction[1] * 1.15, direction[2] * 1.15,
            f'{labels[i]} ($I={eigenvalues[i]:.2f}$)', fontsize=10, color=colors_axes[i])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Inertia Ellipsoid with Principal Axes')

# Equal aspect ratio
max_range = np.max(np.abs(positions_cm)) * 1.2
ax.set_xlim(-max_range, max_range)
ax.set_ylim(-max_range, max_range)
ax.set_zlim(-max_range, max_range)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Decomposition Error vs Rank (CP and Tucker) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CP error vs rank
ax = axes[0]
ax.semilogy(ranks_to_test, cp_final_errors, 'o-', color=COLOR_PRIMARY, linewidth=2, markersize=8)
ax.axhline(noise_ratio, color=COLOR_ACCENT, linestyle='--', linewidth=2, label=f'Noise floor ({noise_ratio:.4e})')
ax.axvline(CP_RANK, color=COLOR_SECONDARY, linestyle=':', linewidth=2, label=f'True rank ({CP_RANK})')
ax.set_xlabel('CP Rank $R$')
ax.set_ylabel('Relative Reconstruction Error')
ax.set_title('CP Decomposition: Error vs Rank')
ax.legend()

# Right: Tucker error vs compression
ax = axes[1]
ax.semilogy(tucker_compression, tucker_errors, 's-', color=COLOR_TERTIARY, linewidth=2, markersize=8)
for i, ranks in enumerate(tucker_rank_configs):
    ax.annotate(f'{ranks}', (tucker_compression[i], tucker_errors[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel('Compression Ratio')
ax.set_ylabel('Relative Reconstruction Error')
ax.set_title('Tucker Decomposition: Error vs Compression')

plt.tight_layout()
plt.show()

In [ ]:
# ---- CP-ALS Convergence Plot ----
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

ax.semilogy(cp_errors, color=COLOR_PRIMARY, linewidth=2)
ax.axhline(noise_ratio, color=COLOR_ACCENT, linestyle='--', linewidth=2, label=f'Noise floor')
ax.set_xlabel('ALS Iteration')
ax.set_ylabel('Relative Reconstruction Error')
ax.set_title(f'CP-ALS Convergence (Rank {CP_RANK})')
ax.legend()

plt.tight_layout()
plt.show()

---
## 10. Extensions

### Tensor Networks

Tensor networks generalize CP and Tucker by representing a high-order tensor as a graph of lower-order tensors connected via contractions. Key variants:

- **Tensor Train (TT)**: chain of 3rd-order tensors $\mathcal{G}_1, \ldots, \mathcal{G}_N$ with
$$\mathcal{X}_{i_1 \ldots i_N} = \mathbf{G}_1(i_1) \cdot \mathbf{G}_2(i_2) \cdots \mathbf{G}_N(i_N)$$
where each $\mathbf{G}_k(i_k)$ is an $r_{k-1} \times r_k$ matrix. This avoids the exponential scaling of Tucker.

- **Hierarchical Tucker**: binary tree structure for logarithmic complexity

### Tensor Completion

Given a partially observed tensor $\mathcal{X}$ with entries $\{x_{ijk} : (i,j,k) \in \Omega\}$, tensor completion recovers the missing entries by assuming low-rank structure:

$$\min_{\mathcal{Y}} \sum_{(i,j,k) \in \Omega} (x_{ijk} - y_{ijk})^2 \quad \text{subject to} \quad \text{rank}(\mathcal{Y}) \leq R$$

This is the tensor analogue of matrix completion (used in recommender systems).

In [ ]:
# ---- Tensor Completion Demo ----
np.random.seed(42)

# Create a low-rank tensor
shape_tc = (8, 10, 12)
true_rank = 3
tc_factors = [np.random.randn(s, true_rank) for s in shape_tc]
X_full = cp_reconstruct(tc_factors).data

# Randomly mask entries
observed_fraction = 0.5
mask = np.random.rand(*shape_tc) < observed_fraction
X_observed = X_full * mask

print(f"Tensor shape: {shape_tc}")
print(f"True rank: {true_rank}")
print(f"Observed entries: {mask.sum()} / {np.prod(shape_tc)} ({100*mask.mean():.1f}%)")
print()

# Simple completion via ALS with masking
def cp_als_masked(X_observed, mask, rank, max_iter=300, tol=1e-8):
    """CP decomposition with missing data via weighted ALS.

    Args:
        X_observed: Observed tensor (zeros for missing). Shape: (I, J, K).
        mask: Binary mask (1 = observed). Shape: (I, J, K).
        rank: Target CP rank. Integer.
        max_iter: Maximum ALS iterations. Integer.
        tol: Convergence tolerance. Scalar.

    Returns:
        factors: List of factor matrices.
        errors: List of reconstruction errors on observed entries.
    """
    shape = X_observed.shape
    N = len(shape)
    factors = [np.random.randn(shape[n], rank) for n in range(N)]
    errors = []

    # Working copy: fill missing entries with current approximation
    X_filled = X_observed.copy()

    for iteration in range(max_iter):
        # Fill missing entries with current reconstruction
        X_approx = cp_reconstruct(factors).data
        X_filled = mask * X_observed + (1 - mask) * X_approx

        # Run one ALS step on the filled tensor
        X_tensor = Tensor(X_filled)
        for n in range(N):
            kr_indices = [i for i in range(N) if i != n]
            kr = factors[kr_indices[-1]]
            for i in reversed(kr_indices[:-1]):
                kr = khatri_rao(factors[i], kr)
            X_n = X_tensor.unfold(n)
            V = kr.T @ kr
            factors[n] = X_n @ kr @ np.linalg.pinv(V)

        # Error on observed entries only
        X_approx = cp_reconstruct(factors).data
        obs_error = np.sqrt(np.sum(mask * (X_full - X_approx)**2) / mask.sum())
        errors.append(obs_error)

        if iteration > 0 and abs(errors[-1] - errors[-2]) < tol:
            break

    return factors, errors


tc_factors_est, tc_errors = cp_als_masked(X_observed, mask, true_rank, max_iter=200)

# Evaluate on missing entries
X_completed = cp_reconstruct(tc_factors_est).data
missing_mask = 1 - mask
rmse_missing = np.sqrt(np.sum(missing_mask * (X_full - X_completed)**2) / missing_mask.sum())
rmse_scale = np.sqrt(np.mean(X_full**2))

print(f"Completion RMSE on missing entries: {rmse_missing:.6e}")
print(f"RMS of true tensor: {rmse_scale:.4f}")
print(f"Normalized RMSE: {rmse_missing / rmse_scale:.4e}")
print(f"Converged in {len(tc_errors)} iterations")

In [ ]:
# ---- Tensor Completion Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

slice_k = 3

# Original
ax = axes[0]
im = ax.imshow(X_full[:, :, slice_k], cmap='RdBu_r', aspect='auto', vmin=-5, vmax=5)
ax.set_title(f'Original $\\mathcal{{X}}(:,:,{slice_k})$')
ax.set_xlabel('Mode 1')
ax.set_ylabel('Mode 0')
plt.colorbar(im, ax=ax, shrink=0.8)

# Observed (with gaps)
ax = axes[1]
observed_slice = X_observed[:, :, slice_k].copy()
observed_slice[~mask[:, :, slice_k].astype(bool)] = np.nan
im = ax.imshow(observed_slice, cmap='RdBu_r', aspect='auto', vmin=-5, vmax=5)
ax.set_title(f'Observed (50% missing)')
ax.set_xlabel('Mode 1')
ax.set_ylabel('Mode 0')
plt.colorbar(im, ax=ax, shrink=0.8)

# Completed
ax = axes[2]
im = ax.imshow(X_completed[:, :, slice_k], cmap='RdBu_r', aspect='auto', vmin=-5, vmax=5)
ax.set_title(f'Completed $\\hat{{\\mathcal{{X}}}}(:,:,{slice_k})$')
ax.set_xlabel('Mode 1')
ax.set_ylabel('Mode 0')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Tensor Completion: Original vs Observed vs Completed', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Tensor Regression

In tensor regression, the predictor is a tensor $\mathcal{X}_i \in \mathbb{R}^{I_1 \times \cdots \times I_N}$ and the response is a scalar $y_i$:

$$y_i = \langle \mathcal{B}, \mathcal{X}_i \rangle + \varepsilon_i$$

where $\mathcal{B}$ is the coefficient tensor. Assuming $\mathcal{B}$ has CP structure reduces the number of parameters from $\prod I_n$ to $R \sum I_n$.

### Connection to Deep Learning

Weight tensors in neural networks (e.g., convolutional filters of shape $C_{\text{out}} \times C_{\text{in}} \times H \times W$) can be compressed via tensor decomposition:

- **CP decomposition** of convolutional layers reduces FLOPs
- **Tucker decomposition** provides more flexible compression
- **Tensor Train** decomposition compresses fully-connected layers

This is an active area of research in efficient neural network inference.

### Key Takeaways

1. **Tensors** generalize matrices to multi-way data; **order**, **fibers**, **slices**, and **unfoldings** are the fundamental building blocks
2. **CP decomposition** expresses a tensor as a sum of rank-1 terms; ALS is the workhorse algorithm
3. **Tucker decomposition** provides a core tensor plus orthogonal factor matrices; HOSVD gives a non-iterative solution
4. **Tensor contraction** generalizes matrix multiplication via Einstein summation
5. **Physical tensors** (inertia, stress, elasticity) and **data tensors** (manipulability, sensor arrays) share the same mathematical framework
6. **Tensor networks** and **completion** extend decompositions to very high orders and missing data